# Section 4 and Section 5 cell block (own CNN, test, augmentation)
Group **PG10** · Neuronauts

**Paste this block into the team notebook after the last Section 3 cell.** Do not repeat Sections 1 to 3.

It uses objects the team notebook already defines: `device`, `SEED`, `set_seed`, `ds`, `relabel`, `class_names`, `to_arrays`, `to_tensor01`, `normalise`, `Xtr`, `ytr`, `make_loaders`, `run_epoch`, `train_model`, `plot_history`, `fit_report`, `count_params`, `RESULTS`, `n_cnn`, plus `np`, `plt`, `torch`, `nn`, `copy`, `time`, `DataLoader`, `TensorDataset`.

Cells marked **DRAFT** hold comment text with [brackets]. Replace every bracket with your own printed numbers and reword it before submitting.

# PART D – Section 4: Own model design

## 4.1 Design

The baseline CNN of Section 2 has 16.8 million parameters, almost all of them in one 16384 → 1024 fully connected layer. It reached 100% training accuracy from epoch 10 while validation accuracy stayed around 60%, so it memorised the training images. Our own network tries to fix that.

* **Two stacked 3×3 convolutions per block.** Two 3×3 layers cover the same area as one 5×5 layer, with fewer weights and one extra ReLU.
* **Channels 32 → 64 → 128 → 256 while the image size halves** (64 → 32 → 16 → 8 → 4). Deeper layers get more channels to hold richer features.
* **BatchNorm after every convolution.** It keeps training stable, so the same learning rate of 0.001 works, and it acts as a mild regulariser.
* **Global average pooling instead of a large fully connected layer.** This removes the 16.8 million parameter layer. Our network has about 1.18 million parameters.
* **Dropout** (0.1 to 0.3 inside the blocks, 0.4 before the output layer), higher in the deeper blocks because those memorise the most.
* **Same training set-up as the baselines** (Adam, lr 0.001, batch size 64, seed 10, no weight decay), so the comparison is fair.

| Stage | Layers | Output | Params |
|---|---|---|---|
| Input | normalised RGB | 3×64×64 | 0 |
| Block 1 | 2 × (Conv3×3 32, BN, ReLU), MaxPool, Dropout2d 0.1 | 32×32×32 | 10,208 |
| Block 2 | 2 × (Conv3×3 64, BN, ReLU), MaxPool, Dropout2d 0.1 | 64×16×16 | 55,552 |
| Block 3 | 2 × (Conv3×3 128, BN, ReLU), MaxPool, Dropout2d 0.2 | 128×8×8 | 221,696 |
| Block 4 | 2 × (Conv3×3 256, BN, ReLU), MaxPool, Dropout2d 0.3 | 256×4×4 | 885,760 |
| Head | GlobalAvgPool, Dropout 0.4, Linear 256 → 10 | 10 | 2,570 |
| **Total** | | | **1,175,786** |

**Worked example (Block 1).** First conv: 3×3×3×32 = 864 weights (no bias, BatchNorm follows). BN: 2×32 = 64 (one scale and one shift per channel; the running averages are not learned). Second conv: 3×3×32×32 = 9,216. BN: 64. Total 864 + 64 + 9,216 + 64 = **10,208**. The head is 256×10 + 10 = **2,570**.

In [ ]:
class OwnNet(nn.Module):
    def __init__(self):
        super().__init__()
        def block(cin, cout, p):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2), nn.Dropout2d(p))
        self.features = nn.Sequential(block(3, 32, 0.1), block(32, 64, 0.1),
                                      block(64, 128, 0.2), block(128, 256, 0.3))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.4), nn.Linear(256, 10))
    def forward(self, x):
        return self.head(self.features(x))

# shape check: follow one dummy image through the network
net_check = OwnNet().eval()
xx = torch.zeros(1, 3, 64, 64)
with torch.no_grad():
    for i, blk in enumerate(net_check.features, 1):
        xx = blk(xx)
        print(f'after block {i}: {tuple(xx.shape)}')
    print('output:', tuple(net_check.head(xx).shape))

n_own = count_params(OwnNet())
assert n_own == 1_175_786
print(f'Own CNN has {n_own:,} parameters = {n_own / n_cnn * 100:.1f}% of the baseline CNN ({n_cnn:,})')

## 4.2 Training (40 epochs, Adam, lr = 0.001, batch size 64, seed 10)

In [ ]:
own_model, own_hist, own_final_state = train_model(OwnNet, 'OWN', epochs=40, lr=1e-3, batch_size=64)

## 4.3 Curves, fitting and comparison with the best baseline

In [ ]:
plot_history(own_hist, 'Own CNN')
fit_report(own_hist, 'Own CNN')

# best baseline = A2 CNN, or the tuned A2 CNN of Section 3 if it scored higher on validation
base_key = 'A2_CNN'
if 'A2_CNN_tuned' in RESULTS and max(RESULTS['A2_CNN_tuned']['val_acc']) > max(RESULTS['A2_CNN']['val_acc']):
    base_key = 'A2_CNN_tuned'
print('Best baseline used for comparison:', base_key)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h in [('Best baseline CNN', RESULTS[base_key]), ('Own CNN', own_hist)]:
    ep = np.arange(1, len(h['val_acc']) + 1)
    a1.plot(ep, h['val_loss'], label=f'{name} val')
    a2.plot(ep, np.array(h['val_acc']) * 100, label=f'{name} val')
    a2.plot(ep, np.array(h['train_acc']) * 100, ls=':', label=f'{name} train')
a1.set(title='Validation loss', xlabel='epoch', ylabel='loss')
a2.set(title='Accuracy (solid = val, dotted = train)', xlabel='epoch', ylabel='accuracy (%)')
for a in (a1, a2):
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Model':<20}{'Params':>14}{'Best val acc':>15}{'Epoch':>8}{'Final gap':>12}{'Time (s)':>10}")
for name, h, n in [('Best baseline CNN', RESULTS[base_key], n_cnn), ('Own CNN', own_hist, n_own)]:
    b = int(np.argmax(h['val_acc']))
    gap = (h['train_acc'][-1] - h['val_acc'][-1]) * 100
    print(f"{name:<20}{n:>14,}{h['val_acc'][b]*100:>14.1f}%{b+1:>8}{gap:>11.1f}%{h['time_s']:>10.0f}")

**DRAFT – fitting and comparison comment (replace every [bracket], reword in your own voice):**

Our own CNN reached its best validation accuracy of [x]% at epoch [n], compared with [y]% for the best baseline CNN. At the end of training the gap between training and validation accuracy is [z] points, against [w] points for the baseline. Validation loss [stays close to its lowest value / starts rising after epoch n], so we would call the model [a fairly good fit / still slightly overfitted from around epoch n]. The baseline was clearly overfitting: 100% training accuracy from epoch 10, with its lowest validation loss at epoch [m]. BatchNorm, dropout and global average pooling reduced the parameters by about 14 times and cut this problem. [One sentence on training time.]

## 4.4 Test evaluation (run ONCE, only after the model and all settings are final)

The test set is the original `valid` split of Tiny ImageNet restricted to our 10 classes (500 images, 50 per class). It was not used for training, model selection or any decision. The guard on the first line stops this cell from being run twice.

In [ ]:
assert 'TEST_DONE' not in globals(), 'The test set was already used once. Do not run this cell again.'

test_full = ds['valid'].filter(lambda y: y in relabel, input_columns='label')
test_full = test_full.map(lambda y: {'label': relabel[y]}, input_columns='label')
X_test, y_test = to_arrays(test_full)
Xte_n = normalise(to_tensor01(X_test))        # training mean and std, as for validation
print(Xte_n.shape, np.bincount(y_test))        # expect 500 images, 50 per class
assert len(y_test) == 500

own_model.eval()
preds = []
with torch.no_grad():
    for i in range(0, len(Xte_n), 256):
        preds.append(own_model(Xte_n[i:i+256].to(device)).argmax(1).cpu())
preds = torch.cat(preds).numpy()
test_acc = (preds == y_test).mean()
TEST_DONE = True

best_val = max(own_hist['val_acc'])
print(f'TEST ACCURACY: {test_acc*100:.1f}%   (best validation accuracy was {best_val*100:.1f}%)')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print(classification_report(y_test, preds, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False)
plt.title('Own CNN: confusion matrix on the 500 test images')
plt.tight_layout(); plt.show()

pairs = [(cm[i, j], class_names[i], class_names[j]) for i in range(10) for j in range(10) if i != j]
print('Most confused pairs (true class -> predicted class):')
for n, t, p in sorted(pairs, reverse=True)[:5]:
    print(f'  {n:2d} images of {t} predicted as {p}')

**DRAFT – test comment (replace every [bracket], reword in your own voice):**

On the 500 unseen test images the model scored [x]%. That is [close to / a bit below] the validation accuracy of [y]%, so choosing the best epoch on validation does not seem to have overfitted the validation set. The best class by F1 was [class] ([f1]) and the weakest was [class] ([f1]). The biggest confusion was [class A] predicted as [class B] ([n] images). [One honest reason, for example they look alike at 64×64 or share a background colour; check it against the Section 1 sample images.] The test set was used once, after the model and all settings were fixed.

# PART E – Section 5 (optional): image augmentation

Same network, same settings, same seed. Only the training data changes: each of the 4,500 training images gets 2 augmented copies (random flip, crop, rotation, colour shift), giving 13,500 training images. The validation set is **not** augmented. The test set is not used in this section.

In [ ]:
from torchvision.transforms import v2 as TV

aug = TV.Compose([TV.RandomHorizontalFlip(0.5),
                  TV.RandomResizedCrop(64, scale=(0.7, 1.0), antialias=True),
                  TV.RandomRotation(15),
                  TV.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)])

set_seed(SEED)
copies = [Xtr] + [torch.stack([aug(im) for im in Xtr]) for _ in range(2)]   # originals + 2 augmented copies
Xtr_aug = normalise(torch.cat(copies).clamp(0, 1))
ytr_aug = torch.cat([ytr] * 3)
print(Xtr_aug.shape, np.bincount(ytr_aug.numpy()))        # 13500 images, 1350 per class

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for r in range(3):
    for j in range(6):
        axes[r, j].imshow(copies[r][j].clamp(0, 1).permute(1, 2, 0)); axes[r, j].axis('off')
axes[0, 0].set_title('row 1 original, rows 2-3 augmented', loc='left', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def train_model_aug(model_fn, name, X, y, epochs, lr, batch_size, seed=SEED):
    set_seed(seed)
    model = model_fn().to(device)
    gen = torch.Generator().manual_seed(seed)
    tr_loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True, generator=gen)
    _, va_loader = make_loaders(batch_size, seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': lr, 'batch_size': batch_size, 'epochs': epochs}
    best_acc, best_state, t0 = -1.0, None, time.time()
    for ep in range(1, epochs + 1):
        tl, ta = run_epoch(model, tr_loader, criterion, optimizer)
        vl, va = run_epoch(model, va_loader, criterion)
        for k, v in zip(['train_loss', 'train_acc', 'val_loss', 'val_acc'], [tl, ta, vl, va]):
            hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        print(f'[{name}] epoch {ep:2d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}')
    hist['time_s'] = time.time() - t0
    RESULTS[name] = hist
    model.load_state_dict(best_state)
    return model, hist

aug_model, aug_hist = train_model_aug(OwnNet, 'OWN_AUG', Xtr_aug, ytr_aug, epochs=40, lr=1e-3, batch_size=64)
plot_history(aug_hist, 'Own CNN + augmentation')
fit_report(aug_hist, 'Own CNN + augmentation')

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h in [('Own CNN (Section 4)', own_hist), ('Own CNN + augmentation', aug_hist)]:
    ep = np.arange(1, len(h['val_acc']) + 1)
    a1.plot(ep, h['val_loss'], label=f'{name} val')
    a2.plot(ep, np.array(h['val_acc']) * 100, label=f'{name} val')
    a2.plot(ep, np.array(h['train_acc']) * 100, ls=':', label=f'{name} train')
a1.set(title='Validation loss', xlabel='epoch'); a2.set(title='Accuracy (%)', xlabel='epoch')
for a in (a1, a2):
    a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{'Model':<26}{'Best val acc':>14}{'Epoch':>8}{'Final gap':>12}{'Time (s)':>10}")
for name, h in [('Own CNN (Section 4)', own_hist), ('Own CNN + augmentation', aug_hist)]:
    b = int(np.argmax(h['val_acc']))
    gap = (h['train_acc'][-1] - h['val_acc'][-1]) * 100
    print(f"{name:<26}{h['val_acc'][b]*100:>13.1f}%{b+1:>8}{gap:>11.1f}%{h['time_s']:>10.0f}")

**DRAFT – Section 5 comment (replace every [bracket], reword in your own voice):**

With augmented data the best validation accuracy was [x]% at epoch [n], against [y]% without augmentation, so it [helped by z points / made little difference / did not help]. The final train–validation gap went from [a] to [b] points. Training accuracy is lower with augmentation because the images are harder to memorise, so we compare on validation accuracy. [One sentence on why: flips, crops and small colour changes show the network each image in different forms.] We did not evaluate the augmented model on the test set.